# Exportar el modelo final
Reentrena **la misma receta ganadora** del proyecto (variante 4b: sobremuestreo con imágenes reales + pesos de clase + ajuste fino) y guarda el modelo para usarlo en la web. Tarda unos 8–10 minutos con GPU T4.

> Activar GPU: Runtime → Change runtime type → T4 GPU.

In [ ]:
import os, subprocess, sys
from pathlib import Path

EN_COLAB = os.path.exists("/content")

if EN_COLAB:
    REPO = "/content/ia-Kaggle-Coffee"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "-b", "main", "https://github.com/JavierAnd0/ia-Kaggle-Coffee", REPO], check=True)
    os.chdir(REPO)

# Subir hasta la carpeta del proyecto (la que tiene src/ y data_splits/), desde donde se abra el notebook.
raiz = Path.cwd()
while not ((raiz / "src").is_dir() and (raiz / "data_splits").is_dir()) and raiz != raiz.parent:
    raiz = raiz.parent
if not (raiz / "src").is_dir():
    raise SystemExit("No encuentro la carpeta del proyecto (con src/ y data_splits/). Abra el notebook dentro del repositorio.")
os.chdir(raiz)
sys.path.insert(0, str(raiz))
print("Proyecto:", raiz)

# Instala solo lo que falte
faltan = []
for modulo, paquete in (("keras_tuner", "keras-tuner"), ("onnx", "onnx"), ("tf2onnx", "tf2onnx"), ("onnxruntime", "onnxruntime"),
                        ("sklearn", "scikit-learn"), ("pandas", "pandas"), ("tensorflow", "tensorflow")):
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)

import pandas as pd, tensorflow as tf
from src import config, generators, tuning, export
from src import train as trainer
N_EXTRA = 150     # imágenes reales repetidas de la clase minoritaria (igual que en la comparación)
EPOCHS, FT_EPOCHS = 30, 10

In [ ]:
tr = pd.read_csv(config.SPLITS_DIR / "train.csv")
va = pd.read_csv(config.SPLITS_DIR / "val.csv")
te = pd.read_csv(config.SPLITS_DIR / "test.csv")
li = generators.make_label_index(tr["label"])
best = tuning.load_best_hp()
minority = tr["label"].value_counts().idxmin()
extra = tr[tr["label"] == minority].sample(N_EXTRA, replace=True, random_state=config.SEED)
tr_aug = pd.concat([tr, extra], ignore_index=True)
print(li, len(tr), '->', len(tr_aug))

In [ ]:
model = trainer.train_model(best, tr_aug, va, li, epochs=EPOCHS, use_class_weight=True,
                            finetune_epochs=FT_EPOCHS, weight_df=tr, verbose=2)

In [ ]:
from sklearn.metrics import classification_report
res = {}
for name, d in (("val", va), ("test", te)):
    ev = trainer.evaluate(model, best, d, li)
    res[name] = {k: round(v, 4) for k, v in ev.items() if k not in ("y_true", "y_pred")}
    print(name, res[name])
ev = trainer.evaluate(model, best, te, li)
print(classification_report(ev['y_true'], ev['y_pred'], target_names=sorted(li, key=li.get), zero_division=0))
print(trainer.confusion_df(ev, li))

In [ ]:
import json
out = export.export_model(model, li, best, res, config.ROOT / "web" / "model")
onnx_path = export.export_onnx(model, out)
print("Diferencia máxima Keras vs ONNX:", export.verificar_onnx(model, onnx_path))   # debe ser ~0
casos = export.casos_paridad(model, best, li, te)
(out / "paridad.json").write_text(json.dumps(casos, indent=2))
print(sorted(p.name for p in out.iterdir()))
zip_path = export.empaquetar(config.ROOT / "modelo_cafe.zip", [config.ROOT / "web" / "model"])
print("Paquete listo:", zip_path)
try:                                   # descarga automática solo en Colab
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Fuera de Colab: el archivo está en esa ruta; copie web/model/ al repositorio.")
